# 1. Setup

In [ ]:
!git clone --branch update-prompts --single-branch https://github.com/longkvy/mura-finance.git

fatal: destination path 'mura-finance' already exists and is not an empty directory.


In [ ]:
%cd /content/mura-finance/

/content/mura-finance


In [3]:
from pathlib import Path

ROOT = Path(".")

# ROOT = Path("/content/mura-finance")
# !pip install -q -r requirements.txt
# !pip install accelerate sentencepiece attrdict tqdm scikit-learn
# print("Requirements installed. Project root:", ROOT)

import sys
import os
import warnings

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))
from src.pipeline.llm_client import LLMClient
from src.pipeline.orchestrator import ReasoningPipeline
from src.evaluation.run_pipeline import run_pipeline_on_sample, pretty_print_metrics
from src.utils.data_loader import load_all_dataframes
from src.evaluation.metrics import compute_classification_metrics

warnings.filterwarnings("ignore")

# 2. Ollama Solution

In [ ]:
!sudo apt update
!sudo apt install -y pciutils zstd systemd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
66 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as re

In [ ]:
import threading
import subprocess
import time


def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"])


thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [1]:
# !ollama pull qwen3
!ollama pull gemma3:1b
# !ollama pull phi4-mini-reasoning
# !ollama pull qwen3:14b
#!ollama pull gemma3:12b

]11;?\pulling manifest ⠙ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 7cd4618c1faf: 100% ▕██████████████████▏ 815 MB                         
pulling e0a42594d802: 100% ▕██████████████████▏  358 B                         
pulling dd084c7d92a3: 100% ▕██████████████████▏ 8.4 KB                         
pulling 3116c5225075: 100% ▕██████████████████▏   77 B                         
pulling 120007c81bf8: 100% ▕██████████████████▏  492 B                         
verifying sha256 digest 
writing manifest 
success 


In [4]:
test_model = "gemma3:1b"

os.environ.setdefault("OLLAMA_HOST", "http://localhost:11434")
os.environ.setdefault("OLLAMA_MODEL", test_model)

'gemma3:1b'

In [ ]:
sys.path.insert(0, str(Path().resolve().parent))

N_SAMPLES = 10
# base = Path("/content/mura-finance")
base = Path().resolve().parent

data = load_all_dataframes(base)

dev = data["dev"]
sample = dev.head(N_SAMPLES).copy()

print(f"Loaded {len(sample)} samples. Columns: text, ticker, true_sentiment")
sample[["text", "ticker", "true_sentiment"]].head(3)

Loaded dev split: 1829 rows from /Users/longnguyen/Desktop/uoa-group1-c6/data/dev.csv
Loaded test split: 462 rows from /Users/longnguyen/Desktop/uoa-group1-c6/data/test.csv
Loaded 10 samples. Columns: text, ticker, true_sentiment


,text,ticker,true_sentiment
0,Economists at Credit Suisse are now neutral o...,EURCHF,Negative
1,New low for the EURUSD as it remains the curre...,EURCHF,Negative
2,EUR/CHF vaults parity for the first time since...,EURCHF,Neutral


### 4 Hops pipeline

In [14]:
llm = LLMClient(max_tokens=1024, model=test_model)
four_hops_pipeline = ReasoningPipeline(llm_client=llm)
y_true_4hop, y_pred_4hop = run_pipeline_on_sample(four_hops_pipeline, sample)

ReasoningContext(text='EURCHF Bias would be for stronger Franc but waiting for clearer SNB monetary policy stance – Credit Suisse', ticker='EURCHF', fx_insight='Okay, here’s a breakdown of the FX signal for EURCHF and YEN (YEN) based on the provided information, focusing solely on the headline and its implications:\n\n**EURCHF**\n\n*   **Currency Affected:** EURCHF\n*   **Direction of Pressure:** Downward\n*   **Rationale:** The headline suggests a stronger Franc, but the wait for SNB’s policy stance indicates uncertainty.  This suggests a potential weakening trend.\n\n**YEN**\n\n*   **Currency Affected:** YEN\n*   **Direction of Pressure:**  No clear pressure detected.\n*   **Rationale:** The headline focuses on SNB’s policy, not a specific direction for the YEN.\n\n\n---\n\n**Important Disclaimer:** *This analysis is based solely on the provided headline and limited context.  FX markets are complex and influenced by numerous factors. This is a preliminary assessment and should not be

In [15]:
metrics_4hop = compute_classification_metrics(y_true_4hop, y_pred_4hop)
pretty_print_metrics(
    f"4-Hop Pipeline (same {N_SAMPLES} samples)", metrics_4hop, y_true_4hop, y_pred_4hop
)

4-Hop Pipeline (same 10 samples) — Metrics
--------------------------------------------------
  n (valid pairs): 10
  Accuracy:        0.6000
  F1 (macro):      0.2500
  Precision (macro): 0.2000
  Recall (macro):    0.3333

Confusion matrix (rows=true, cols=pred):
          Negative  Neutral  Positive
Negative         0        0         2
Neutral          0        0         2
Positive         0        0         6


### Single Prompt Pipeline

In [16]:
llm = LLMClient(max_tokens=1024, model=test_model)
single_pipeline = ReasoningPipeline(llm_client=llm, mode="single")
y_true_single, y_pred_single = run_pipeline_on_sample(single_pipeline, sample)

ReasoningContext(text='EURCHF Bias would be for stronger Franc but waiting for clearer SNB monetary policy stance – Credit Suisse', ticker='EURCHF', fx_insight=None, base_sentiment=None, quote_sentiment=None, sentiment='Neutral', hop_results={'simple_prompt': {'sentiment': 'Neutral'}}, raw_responses={'simple_prompt': 'Neutral'})
ReasoningContext(text='New lows for the EURUSD. EURCHF down as well and tests its 200 day MA.', ticker='EURCHF', fx_insight=None, base_sentiment=None, quote_sentiment=None, sentiment='Negative', hop_results={'simple_prompt': {'sentiment': 'Negative'}}, raw_responses={'simple_prompt': 'Negative'})
ReasoningContext(text='Does a jump in EURCHF point to a break above 108 in EURUSD – SocGen', ticker='EURCHF', fx_insight=None, base_sentiment=None, quote_sentiment=None, sentiment='Positive', hop_results={'simple_prompt': {'sentiment': 'Positive'}}, raw_responses={'simple_prompt': 'Positive'})
ReasoningContext(text='USDCHF stalls its run higher at the 200 bar MA on the

In [17]:
metrics_single = compute_classification_metrics(y_true_single, y_pred_single)
pretty_print_metrics(
    "Sinple Pipeline (same {N_SAMPLES} samples)",
    metrics_single,
    y_true_single,
    y_pred_single,
)

Sinple Pipeline (same {N_SAMPLES} samples) — Metrics
--------------------------------------------------
  n (valid pairs): 10
  Accuracy:        0.8000
  F1 (macro):      0.6966
  Precision (macro): 0.7857
  Recall (macro):    0.6667

Confusion matrix (rows=true, cols=pred):
          Negative  Neutral  Positive
Negative         1        1         0
Neutral          0        1         1
Positive         0        0         6


# 3. Flan T5 XXL

In [19]:
sys.path.insert(0, str(Path().resolve().parent))

N_SAMPLES = 10
# base = Path("/content/mura-finance")
base = Path().resolve().parent

data = load_all_dataframes(base)

dev = data["dev"]
sample = dev.head(N_SAMPLES).copy()

print(f"Loaded {len(sample)} samples. Columns: text, ticker, true_sentiment")
sample[["text", "ticker", "true_sentiment"]].head(3)

Loaded dev split: 1829 rows from /Users/longnguyen/Desktop/uoa-group1-c6/data/dev.csv
Loaded test split: 462 rows from /Users/longnguyen/Desktop/uoa-group1-c6/data/test.csv
Loaded 10 samples. Columns: text, ticker, true_sentiment


,text,ticker,true_sentiment
0,Economists at Credit Suisse are now neutral o...,EURCHF,Negative
1,New low for the EURUSD as it remains the curre...,EURCHF,Negative
2,EUR/CHF vaults parity for the first time since...,EURCHF,Neutral


### 4 hops pipeline

In [ ]:
llm = LLMClient(
    provider="flan_t5",
    model="google/flan-t5-xxl",
    max_tokens=2048,
    temperature=0.0,
    device=None,
)

flan_t5_4hops_pipeline = ReasoningPipeline(llm_client=llm)

y_true_flan_4hop, y_pred_flan_4hop = run_pipeline_on_sample(
    flan_t5_4hops_pipeline, sample
)

Provider: flan_t5
Model: google/flan-t5-xxl (loads on first generate)


In [ ]:
metrics_flan_4hop = compute_classification_metrics(y_true_flan_4hop, y_pred_flan_4hop)
pretty_print_metrics(
    f"4-Hop Pipeline (same {N_SAMPLES} samples)",
    flan_t5_4hops_pipeline,
    y_true_flan_4hop,
    y_pred_flan_4hop,
)